# Reproducible classification of FRP shear-failure mode (D/R)

**Dataset:** `FRP_Dataset_v1_FINAL_VALIDATED_LOCKED.xlsx`  
**Frozen sheet:** `Dataset_v1_FROZEN`  
**Primary target:** `FM` (D/R)  
**Validation:** nested group-aware cross-validation with **Leave-One-Study-Out (LOSO)** external testing.

This notebook is designed for independent review and reproduction in **Google Colab** or **VS Code/Jupyter**. It never writes to the source Excel workbook.

## Experimental rules fixed before model fitting
- 53 specimens from 8 independent studies (`paper_ref`).
- Compare X1 (17 physical variables) against X2 (13-variable reduced representation).
- External validation: one complete study held out at a time.
- Hyperparameter selection uses only the remaining studies through `GroupKFold`.
- `WC` is one-hot encoded inside each training pipeline.
- Scaling is fitted inside the pipeline, never globally.
- Class imbalance is handled with class weighting where supported; no SMOTE in this primary experiment.
- Primary metrics: balanced accuracy, macro-F1, and MCC.
- Complementary metrics: accuracy, class-wise precision/recall/specificity, ROC-AUC, and PR-AUC when mathematically defined.
- Predictions, fold results, hyperparameters, confusion matrices, and environment metadata are exported.

## 0. Reproduction instructions

### Google Colab
1. Open this notebook in Colab.
2. Run the installation cell below.
3. Upload the validated Excel file when prompted, or place it in `/content/`.
4. Run **Runtime → Run all**.
5. Download the generated `classification_results/` folder (or its ZIP archive).

### VS Code / local Jupyter
Recommended: Python **3.11 or 3.12**, VS Code, the Microsoft **Python** and **Jupyter** extensions, and a virtual environment. Put this notebook and the Excel file in the same project folder, install the dependencies, select that Python kernel, and run all cells from top to bottom.

> Reproducibility note: exact package versions used in each run are written automatically to `environment_versions.txt`.

In [ ]:
# 1. Install dependencies (safe in Colab; in a local venv you can run the equivalent pip command once).
# Uncomment the next line if packages are not already installed.
# %pip install -q pandas numpy openpyxl scikit-learn matplotlib xgboost joblib

In [ ]:
# 2. Imports and deterministic configuration
from pathlib import Path
import json, platform, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (balanced_accuracy_score, f1_score, matthews_corrcoef,
    accuracy_score, precision_score, recall_score, confusion_matrix,
    roc_auc_score, average_precision_score, ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_sample_weight

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as e:
    XGB_AVAILABLE = False
    print('XGBoost unavailable:', e)

RANDOM_STATE = 20261001
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore', category=UserWarning)

OUT = Path('classification_results')
CM_DIR = OUT / 'confusion_matrices'
OUT.mkdir(exist_ok=True)
CM_DIR.mkdir(exist_ok=True)
print('Python:', sys.version.split()[0], '| scikit-learn:', sklearn.__version__, '| XGBoost:', XGB_AVAILABLE)

In [ ]:
# 3. Locate the source workbook. The source is READ ONLY in this workflow.
CANDIDATES = [
    Path('FRP_Dataset_v1_FINAL_VALIDATED_LOCKED.xlsx'),
    Path('FRP_Dataset_v1_FINAL_VALIDATED_LOCKED(1).xlsx'),
    Path('/content/FRP_Dataset_v1_FINAL_VALIDATED_LOCKED.xlsx'),
    Path('/content/FRP_Dataset_v1_FINAL_VALIDATED_LOCKED(1).xlsx'),
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    # Colab-only upload fallback.
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError('No workbook uploaded.')
        DATA_PATH = Path(next(iter(uploaded.keys())))
    except ImportError:
        raise FileNotFoundError('Place the validated Excel workbook beside this notebook and rerun this cell.')

SHEET = 'Dataset_v1_FROZEN'
df = pd.read_excel(DATA_PATH, sheet_name=SHEET, engine='openpyxl')
print('Loaded:', DATA_PATH)
print('Shape:', df.shape)

In [ ]:
# 4. Integrity checks: stop immediately if the frozen dataset is not the expected one.
X1 = ['bw_mm','d_mm','df_mm','a_d','fc_MPa','fsy_MPa','rho_s','fvy_MPa','rho_sv',
      'tf_mm','wf_mm','sf_mm','ffu_MPa','Ef_GPa','rho_fv','alpha_f_deg','WC']
X2 = ['bw_mm','df_mm','a_d','fc_MPa','fsy_MPa','rho_s','fvy_MPa','rho_sv',
      'ffu_MPa','Ef_GPa','rho_fv','alpha_f_deg','WC']
REQUIRED = sorted(set(X1 + X2 + ['paper_ref','specimen','FM','Vf_kN']))
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f'Missing required columns: {missing}'
assert len(df) == 53, f'Expected 53 specimens, found {len(df)}.'
assert df['paper_ref'].nunique() == 8, f'Expected 8 studies, found {df.paper_ref.nunique()}.'
assert set(df['FM'].dropna().unique()) == {'D','R'}, f'Unexpected FM labels: {df.FM.unique()}'
assert df[X1 + ['FM','paper_ref']].isna().sum().sum() == 0, 'Missing values detected in modeling fields.'
assert not df[['paper_ref','specimen']].duplicated().any(), 'Duplicate (paper_ref, specimen) detected.'
print('Integrity checks PASSED.')
print(df['FM'].value_counts())
print('Studies:', sorted(df['paper_ref'].unique().tolist()))

In [ ]:
# 5. Audit class distribution by study. Single-class external folds are expected and are NOT an error.
class_by_study = pd.crosstab(df['paper_ref'], df['FM']).reindex(columns=['D','R'], fill_value=0)
class_by_study['N'] = class_by_study.sum(axis=1)
display(class_by_study)
class_by_study.to_csv(OUT / 'class_distribution_by_study.csv')

## 6. Preprocessing
`WC` is nominal and is therefore one-hot encoded. All other X1/X2 fields are numeric. Scaling is applied only to models that are sensitive to feature scale. Crucially, transformers are fitted inside each outer training fold, preventing leakage from the held-out study.

In [ ]:
# 6. Pipeline builders
def make_preprocessor(features, scale_numeric=True):
    cat = ['WC']
    num = [c for c in features if c not in cat]
    num_pipe = Pipeline([('scaler', StandardScaler())]) if scale_numeric else 'passthrough'
    return ColumnTransformer([
        ('num', num_pipe, num),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop=None), cat)
    ], remainder='drop')

def model_specs(features):
    specs = {
        'Dummy': (DummyClassifier(strategy='prior'), {}, False),
        'Logistic': (LogisticRegression(max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE),
                     {'model__C':[0.1,1.0,10.0]}, True),
        'LDA': (LinearDiscriminantAnalysis(),
                {'model__solver':['lsqr'], 'model__shrinkage':['auto',0.1,0.5]}, True),
        'SVM_RBF': (SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
                    {'model__C':[0.5,1,5], 'model__gamma':['scale',0.01,0.1]}, True),
        'KNN': (KNeighborsClassifier(),
                {'model__n_neighbors':[3,5,7], 'model__weights':['uniform','distance']}, True),
        'RandomForest': (RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
                         {'model__max_depth':[None,3,5], 'model__min_samples_leaf':[1,2,4]}, False),
    }
    if XGB_AVAILABLE:
        specs['XGBoost'] = (XGBClassifier(n_estimators=200, objective='binary:logistic', eval_metric='logloss',
                                          random_state=RANDOM_STATE, n_jobs=-1),
                            {'model__max_depth':[2,3], 'model__learning_rate':[0.03,0.1],
                             'model__subsample':[0.8,1.0], 'model__colsample_bytree':[0.8,1.0]}, False)
    return specs

## 7. Nested LOSO engine
For each outer fold, one complete `paper_ref` is untouched until final prediction. Hyperparameters are selected only among the remaining studies using `GroupKFold`. Balanced accuracy is the tuning score. XGBoost receives a fold-specific `scale_pos_weight`; other supported classifiers use balanced class weights. KNN has no native class weighting, which is documented rather than hidden.

In [ ]:
# 7. Metrics and nested LOSO execution
def binary_metrics(y_true, y_pred, score=None):
    # R is treated as the positive class only for binary precision/recall/specificity reporting.
    yb = (pd.Series(y_true).to_numpy() == 'R').astype(int)
    pb = (pd.Series(y_pred).to_numpy() == 'R').astype(int)
    tn, fp, fn, tp = confusion_matrix(yb, pb, labels=[0,1]).ravel()
    out = {
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_R': precision_score(yb, pb, zero_division=0),
        'recall_R': recall_score(yb, pb, zero_division=0),
        'specificity_D': tn/(tn+fp) if (tn+fp) else np.nan,
    }
    if score is not None and len(np.unique(yb)) == 2:
        out['roc_auc'] = roc_auc_score(yb, score)
        out['pr_auc'] = average_precision_score(yb, score)
    else:
        out['roc_auc'] = np.nan
        out['pr_auc'] = np.nan
    return out

def positive_score(estimator, X):
    if hasattr(estimator, 'predict_proba'):
        probs = estimator.predict_proba(X)
        classes = list(estimator.classes_)
        return probs[:, classes.index('R')]
    if hasattr(estimator, 'decision_function'):
        s = estimator.decision_function(X)
        classes = list(estimator.classes_)
        return s if classes[-1] == 'R' else -s
    return np.full(len(X), np.nan)

def run_nested_loso(features, feature_set_name):
    X = df[features].copy(); y = df['FM'].copy(); groups = df['paper_ref'].copy()
    logo = LeaveOneGroupOut()
    predictions, folds, params = [], [], []

    for model_name, (base_model, grid, scale) in model_specs(features).items():
        print(f'[{feature_set_name}] {model_name}')
        for fold_no, (tr, te) in enumerate(logo.split(X, y, groups), start=1):
            Xtr, Xte = X.iloc[tr], X.iloc[te]
            ytr, yte = y.iloc[tr], y.iloc[te]
            gtr = groups.iloc[tr]
            held_out = groups.iloc[te].iloc[0]

            model = clone(base_model)
            # XGBoost needs numeric labels and a fold-specific imbalance weight.
            is_xgb = model_name == 'XGBoost'
            if is_xgb:
                nD, nR = (ytr=='D').sum(), (ytr=='R').sum()
                model.set_params(scale_pos_weight=nD/max(nR,1))

            pipe = Pipeline([('prep', make_preprocessor(features, scale)), ('model', model)])
            inner_splits = min(5, gtr.nunique())
            inner = GroupKFold(n_splits=inner_splits)

            ytr_fit = (ytr=='R').astype(int) if is_xgb else ytr
            yte_for_pred = yte

            if grid:
                search = GridSearchCV(pipe, grid, scoring='balanced_accuracy', cv=inner,
                                      n_jobs=-1, refit=True, error_score=np.nan)
                search.fit(Xtr, ytr_fit, groups=gtr)
                fitted = search.best_estimator_
                best = search.best_params_
            else:
                fitted = pipe.fit(Xtr, ytr_fit)
                best = {}

            raw_pred = fitted.predict(Xte)
            if is_xgb:
                ypred = np.where(np.asarray(raw_pred).astype(int)==1, 'R', 'D')
                score = fitted.predict_proba(Xte)[:,1]
            else:
                ypred = raw_pred
                score = positive_score(fitted, Xte)

            fold_m = binary_metrics(yte_for_pred, ypred, score)
            fold_m.update({'feature_set':feature_set_name,'model':model_name,'fold':fold_no,
                           'held_out_study':held_out,'n_test':len(te)})
            folds.append(fold_m)
            params.append({'feature_set':feature_set_name,'model':model_name,'fold':fold_no,
                           'held_out_study':held_out,'best_params':json.dumps(best, sort_keys=True)})

            for idx, pred, sc in zip(te, ypred, score):
                predictions.append({
                    'feature_set':feature_set_name, 'model':model_name,
                    'paper_ref':df.loc[idx,'paper_ref'], 'specimen':df.loc[idx,'specimen'],
                    'FM_real':df.loc[idx,'FM'], 'FM_pred':pred, 'prob_R':float(sc) if np.isfinite(sc) else np.nan
                })

    return pd.DataFrame(predictions), pd.DataFrame(folds), pd.DataFrame(params)

In [ ]:
# 8. Execute the complete experiment: 7 models x X1/X2 when XGBoost is available.
p1, f1, h1 = run_nested_loso(X1, 'X1_full')
p2, f2, h2 = run_nested_loso(X2, 'X2_reduced')
predictions = pd.concat([p1,p2], ignore_index=True)
fold_results = pd.concat([f1,f2], ignore_index=True)
hyperparameters = pd.concat([h1,h2], ignore_index=True)
print('Predictions generated:', len(predictions))

In [ ]:
# 9. Global out-of-study metrics across all 53 external predictions for each model/feature set.
summary_rows=[]
for (fs, model), g in predictions.groupby(['feature_set','model'], sort=False):
    assert len(g)==53, f'{fs}/{model}: expected 53 external predictions, got {len(g)}'
    m = binary_metrics(g['FM_real'], g['FM_pred'], g['prob_R'].to_numpy())
    m.update({'feature_set':fs,'model':model,'N':len(g)})
    summary_rows.append(m)
summary = pd.DataFrame(summary_rows).sort_values(['feature_set','balanced_accuracy','macro_f1'], ascending=[True,False,False])
display(summary.round(4))

In [ ]:
# 10. Confusion matrices from the 53 concatenated LOSO predictions.
for (fs, model), g in predictions.groupby(['feature_set','model'], sort=False):
    fig, ax = plt.subplots(figsize=(4.5,4.0))
    ConfusionMatrixDisplay.from_predictions(g['FM_real'], g['FM_pred'], labels=['D','R'], cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(f'{model} — {fs} — LOSO')
    fig.tight_layout()
    fig.savefig(CM_DIR / f'{fs}__{model}.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
print('Saved confusion matrices to:', CM_DIR)

In [ ]:
# 11. Per-study descriptive performance. ROC/PR may be undefined in single-class held-out studies.
by_study=[]
for (fs, model, study), g in predictions.groupby(['feature_set','model','paper_ref'], sort=False):
    m=binary_metrics(g['FM_real'],g['FM_pred'],g['prob_R'].to_numpy())
    m.update({'feature_set':fs,'model':model,'paper_ref':study,'N':len(g),
              'classes_in_test':','.join(sorted(g.FM_real.unique()))})
    by_study.append(m)
by_study=pd.DataFrame(by_study)
display(by_study.head())

In [ ]:
# 12. Export auditable outputs.
summary.to_csv(OUT/'classification_summary.csv', index=False)
predictions.to_csv(OUT/'classification_predictions.csv', index=False)
fold_results.to_csv(OUT/'classification_by_fold.csv', index=False)
by_study.to_csv(OUT/'classification_by_study.csv', index=False)
hyperparameters.to_csv(OUT/'best_hyperparameters.csv', index=False)

versions = {
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__,
    'xgboost': getattr(xgb,'__version__','not installed') if XGB_AVAILABLE else 'not installed',
    'random_state': RANDOM_STATE,
    'source_file': str(DATA_PATH),
    'sheet': SHEET,
    'n_rows': len(df),
    'n_studies': int(df.paper_ref.nunique()),
    'X1': X1,
    'X2': X2,
}
(OUT/'environment_versions.txt').write_text(json.dumps(versions, indent=2, default=str), encoding='utf-8')
print('All outputs written to:', OUT.resolve())

In [ ]:
# 13. Optional: create one ZIP file for sharing/review.
import shutil
zip_path = shutil.make_archive('classification_results', 'zip', root_dir=OUT)
print('Created:', zip_path)

## Interpretation rules for reviewers
1. Treat **balanced accuracy, macro-F1 and MCC** as the primary comparison metrics; raw accuracy is secondary because D/R is imbalanced.
2. Do not select a model from in-sample performance. Only the concatenated LOSO predictions are used for primary reporting.
3. Compare X1 and X2 under the same external folds. A reduced feature set is considered useful only if it maintains or improves out-of-study generalization.
4. Inspect study-level results because several held-out studies contain only one observed class. This is a property of the evidence base, not a reason to reshuffle specimens across studies.
5. Do not alter or delete statistically extreme specimens merely to improve classification performance; the source dataset is frozen and independently validated.